# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: d:\Facultate\ADC\AI_Engineering\echochamber-project-team-1
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [3]:
student_id = "student_01"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [4]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [18]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [10]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(15)

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64

In [11]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(15)

,source_channel,video_title,text
10488,RecorderRomania,EXPLICATIV RECORDER: Cazul Gânj. Cum a devenit...,Oare de ce nu sunt surprins? Nu au invățat din...
1861,georgesimionoficial,Așa a fost astăzi la Tânjaua Hotenarilor în Ma...,Foarte fain votez cu respect pe domnu Simion P...
13796,RecorderRomania,Investigație Recorder: Cea mai mare firmă-fant...,Un mic grup de jurnalisti face ce nu poate sta...
4022,digi24hd56,🟣 Știrile Digi24 de la ora 17 – 20 martie 2026,Ati ajuns de toata jena.Cum ma sa le dati atat...
18027,RecorderRomania,PORTRET DE CANDIDAT: Nicușor Dan,Dintre toti candidatii care s-au perindat din ...
29424,AdevaruriSecrete,Predictiile The Simpsons Pentru APRILIE 2025 V...,"Din câte am înțeles eu , produsele care vin di..."
20186,CălinGeorgescu-CanalulOficial,Călin Georgescu - O nouă categorie de păduchi ...,Va iubesc domnul Calin Georgescu ❤🇹🇩 sunteti p...
9315,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,"Mă tem, mă tem cum nu am făcut-o vreodată. Pri..."
14194,RecorderRomania,România lui Iliescu,"Nu degeaba, numele imnului României este: ""Deș..."
27095,turcescu111,Psihiatria salvează România!,"Păi noi spunem de foarte mult timp, ca Harpale..."


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [14]:
# eșantionare aleatorie fără seed fix
sample_df = df.sample(n=10).copy()

sample_df[["source_channel", "text"]]

,source_channel,text
11850,RecorderRomania,Multi prosti pe lumea asta! pacat de cei care ...
27653,turcescu111,EU CÎND AUD CE SPUI!ROBERT! DIN PRIMELE CUVINT...
3172,georgesimionoficial,"Salut Frate, noi suntem alături de tine.... Ne..."
14588,RecorderRomania,Material bun pe partea de jurnalism investigat...
993,georgesimionoficial,Respect George Simion. Nevoile românilor trebu...
13900,RecorderRomania,"Născut în 96, declarațiile de la sfârșitul vid..."
6142,@CălinGeorgescu-CanalulOficial,Jigodiile astea își bat joc de poporul român.....
3443,georgesimionoficial,FELICITARI GEORGE ! ADEVĂRAT ! CALIN GEORGESCU...
11148,RecorderRomania,"E bine de stiut, deci de 2 ori pe saptamana eu..."
13007,RecorderRomania,Boomerii sunt cei mai traumatizați și sunt cei...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [15]:
SYSTEM_PROMPT = """
Ești un model de adnotare pentru analiză de discurs politic.
Sarcina ta este să extragi informații structurale din comentarii politice.
Returnează doar JSON valid.
Nu explica deciziile și nu adăuga text suplimentar.
Nu inventa informații care nu apar explicit în comentariu.
"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu politic și identifică:

1. target:
persoana, instituția, grupul sau actorul politic principal vizat de comentariu.
Dacă nu există, folosește "none".

2. stance:
poziția față de target:
- pro
- anti
- neutru
- ambiguu
- none

3. sentiment:
sentimentul dominant:
- pozitiv
- negativ
- mixt
- neutru

4. tone:
modul dominant de formulare:
- acuzator
- ironic
- mobilizator
- defensiv
- afectiv
- informativ
- neutru

5. topic:
tema principală a comentariului:
- alegeri
- justiție
- corupție
- geopolitică
- economie
- identitate națională
- religie
- media
- sănătate
- altele

6. interpretation_problem:
tipul principal de interpretare prezent:
- conspiraționist
- anti-elitist
- suveranist
- moral-religios
- instituțional
- none

Important:
- Codează doar informațiile prezente în comentariu.
- Nu folosi informații externe.
- Dacă textul este ironic, codează sensul intenționat.
- Returnează doar JSON valid.
- Nu adăuga explicații.

Returnează JSON valid cu exact aceste chei:
target, stance, sentiment, tone, topic, interpretation_problem

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [21]:
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [22]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [23]:
n_comments = 10  # schimbă aici: 3, 5 sau 10
sample_for_prompt = sample_df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,RecorderRomania,EXPLICATIV RECORDER. Cum încearcă Rusia să ne ...,Multi prosti pe lumea asta! pacat de cei care ...,"```json\n{\n ""target"": ""populația"",\n ""stanc..."
1,turcescu111,"ATENȚIE, ne FURĂ! Marea hoție cu prețul la pompă",EU CÎND AUD CE SPUI!ROBERT! DIN PRIMELE CUVINT...,"```json\n{\n ""target"": ""conducătorii actuali ..."
2,georgesimionoficial,Țara are nevoie de toți românii!,"Salut Frate, noi suntem alături de tine.... Ne...","```json\n{\n ""target"": ""Țara Românească"",\n ..."
3,RecorderRomania,Investigație cu camera ascunsă. Cât de ușor aj...,Material bun pe partea de jurnalism investigat...,"```json\n{\n ""target"": ""Nicoșor Dan"",\n ""st..."
4,georgesimionoficial,#unitate #democratie #georgesimion #impreuna #...,Respect George Simion. Nevoile românilor trebu...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
5,RecorderRomania,România lui Iliescu,"Născut în 96, declarațiile de la sfârșitul vid...","```json\n{\n ""target"": ""Ion, regimul comunist..."
6,@CălinGeorgescu-CanalulOficial,Călin Georgescu - Revoluția adevărului ( 30.12...,Jigodiile astea își bat joc de poporul român.....,"```json\n{\n ""target"": ""guvern"",\n ""stance"":..."
7,georgesimionoficial,"Romania are un președinte, numele lui este CAL...",FELICITARI GEORGE ! ADEVĂRAT ! CALIN GEORGESCU...,"```json\n{\n ""target"": ""Călin Georgescu"",\n ..."
8,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"E bine de stiut, deci de 2 ori pe saptamana eu...","```json\n{\n ""target"": ""none"",\n ""stance"": ""..."
9,RecorderRomania,DOCUMENTAR RECORDER. Singuri,Boomerii sunt cei mai traumatizați și sunt cei...,"```json\n{\n ""target"": ""Boomerii"",\n ""stance..."


# 9. Verificam rezultatele

In [24]:
results_df.model_output[0]

'```json\n{\n  "target": "populația",\n  "stance": "anti",\n  "sentiment": "negativ",\n  "tone": "acuzator",\n  "topic": "altele",\n  "interpretation_problem": "none"\n}\n```'

In [25]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [26]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,RecorderRomania,EXPLICATIV RECORDER. Cum încearcă Rusia să ne ...,Multi prosti pe lumea asta! pacat de cei care ...,populația,anti,negativ,acuzator,altele,none,
1,turcescu111,"ATENȚIE, ne FURĂ! Marea hoție cu prețul la pompă",EU CÎND AUD CE SPUI!ROBERT! DIN PRIMELE CUVINT...,conducătorii actuali ai României,anti,negativ,acuzator,altele,anti-elitist,
2,georgesimionoficial,Țara are nevoie de toți românii!,"Salut Frate, noi suntem alături de tine.... Ne...",Țara Românească,pro,pozitiv,mobilizator,identitate națională,none,
3,RecorderRomania,Investigație cu camera ascunsă. Cât de ușor aj...,Material bun pe partea de jurnalism investigat...,Nicoșor Dan,anti,negativ,acuzator,corupție,none,
4,georgesimionoficial,#unitate #democratie #georgesimion #impreuna #...,Respect George Simion. Nevoile românilor trebu...,George Simion,pro,pozitiv,afectiv,identitate națională,none,
5,RecorderRomania,România lui Iliescu,"Născut în 96, declarațiile de la sfârșitul vid...","Ion, regimul comunist, societatea românească",anti,negativ,afectiv,corupție,anti-elitist,
6,@CălinGeorgescu-CanalulOficial,Călin Georgescu - Revoluția adevărului ( 30.12...,Jigodiile astea își bat joc de poporul român.....,guvern,anti,negativ,acuzator,altele,anti-elitist,
7,georgesimionoficial,"Romania are un președinte, numele lui este CAL...",FELICITARI GEORGE ! ADEVĂRAT ! CALIN GEORGESCU...,Călin Georgescu,pro,pozitiv,mobilizator,alegeri,none,
8,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"E bine de stiut, deci de 2 ori pe saptamana eu...",none,none,negativ,acuzator,altele,none,
9,RecorderRomania,DOCUMENTAR RECORDER. Singuri,Boomerii sunt cei mai traumatizați și sunt cei...,Boomerii,anti,negativ,acuzator,altele,none,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [27]:
# salvare rezultate ca CSV
results_df.to_csv("annotated_comments.csv", index=False, encoding="utf-8-sig")

print("CSV salvat")

CSV salvat




Promptul a funcționat bine în comentariile cu target clar și poziționare explicită, mai ales în mesajele pro/anti față de politicieni, justiție sau instituții. Aici, stance-ul și tonul au fost identificate relativ corect.

Totuși, promptul eșuează mai ales la nivelul comentariilor ironice, conspiraționiste sau cu mai multe target-uri. Uneori modelul a inventat target-uri prea generale sau a ales greșit actorul principal.

În anumite cazuri a confundat sentimentul negativ cu stance-ul anti, mai ales în comentariile foarte scurte sau agresive.

Sarcasmul, ambiguitatea și prezența mai multor actori politici au creat probleme frecvente, deoarece modelul a avut dificultăți în identificarea sensului dominant și a target-ului principal.

În următoarea versiune a promptului aș adăuga reguli mai stricte pentru identificarea target-ului, exemple pentru diferența dintre sentiment și stance și instrucțiuni speciale pentru comentarii ironice sau cu multiple ținte.